<p style="text-align:center;">
    <img width="400" alt="Flotação catiônica reversa de minério de ferro" src="https://github.com/ginsaputra/gangue-forecast-in-flotation-concentrate/blob/main/reverse-cationic-flotation-iron-silica.png?raw=true">


Este notebook explora a aplicação de *Deep Learning* para prever a ganga (*%sílica*) no concentrado de flotação. A previsão ajudará os engenheiros de processo a avaliar a pureza do concentrado de flotação e tomar ações corretivas com antecedência. Mais especificamente, o objetivo é responder às seguintes perguntas:
- Com quantos passos (horas) de antecedência o *%sílica no concentrado* pode ser previsto?
- É possível prever o *%sílica no concentrado* sem usar os dados do *%ferro no concentrado*?

In [ ]:
# Importando as bibliotecas relevantes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Dados de Flotação

O conjunto de dados foi obtido de uma planta de processamento mineral que separa sílica do minério de ferro usando o método de flotação catiônica reversa. Dados contínuos do processo foram coletados da 1:00 do dia 10 de Março de 2017 até as 23:00 do dia 9 de Setembro de 2017.

Cada linha de dados consiste em 23 medições que podem ser categorizadas em quatro tipos:
- matérias-primas (colunas 2-3);
- variáveis de ambiente (colunas 4-8);
- variáveis de processo (colunas 9-22);
- materiais processados (colunas 23-24).

As matérias-primas e os materiais processados foram amostrados de hora em hora, enquanto as outras variáveis foram amostradas a cada 20 segundos.

In [ ]:
filepath = '../input/quality-prediction-in-a-mining-process/'
filename = 'MiningProcess_Flotation_Plant_Database.csv'
cols_renamed = [
    'date',          # Timestamp das medições, formato AAAA-MM-DD HH:MM:SS
    'feed_iron',     # %Ferro (valioso) no minério sendo alimentado na célula de flotação
    'feed_silica',   # %Sílica (ganga) no minério sendo alimentado na célula
    'starch_flow',   # Quantidade de amido (reagente) adicionado na célula, em m^3/h
    'amina_flow',    # Quantidade de amina (reagente) adicionada na célula, em m^3/h
    'pulp_flow',     # Quantidade de polpa de minério alimentada na célula, em toneladas/hora
    'pulp_ph',       # Acidez/alcalinidade da polpa de minério em uma escala de 0-14
    'pulp_density',  # Quantidade de minério na polpa, entre 1-3 kg/cm^3
    'air_col1',      # Volume de ar injetado na célula, medido em Nm3/h
    'air_col2',
    'air_col3',
    'air_col4',
    'air_col5',
    'air_col6',
    'air_col7',
    'level_col1',    # Altura da espuma na célula, medida em mm
    'level_col2',
    'level_col3',
    'level_col4',
    'level_col5',
    'level_col6',
    'level_col7',
    'conc_iron',     # Medição em laboratório: %Ferro no final do processo de flotação
    'conc_silica']   # Medição em laboratório: %Sílica no final do processo de flotação

df = pd.read_csv(
    filepath+filename,
    header=0,
    names=cols_renamed,
    parse_dates=['date'],
    infer_datetime_format=True,
    decimal=',')
df.head()

In [ ]:
df.info()

# Pré-processamento dos Dados de Séries Temporais

A coluna `date` é reamostrada para reduzir a frequência de nossos dados de série temporal para uma base horária. Isso é feito selecionando apenas a primeira medição de cada hora, simplificando os dados sem perder o comportamento geral.

In [ ]:
# Reamostrando os dados para uma base horária
df = df.set_index('date').resample('H').first()
df.shape

Ao inspecionar os dados, são encontradas 318 linhas contendo valores nulos (*missing values*) entre `2017-03-16 06:00` e `2017-03-29 11:00`. Valores ausentes introduzem descontinuidade na série temporal e podem ser prejudiciais para a previsão do modelo. Portanto, apenas os dados a partir de `2017-03-29 12:00` serão utilizados.

In [ ]:
nans = df[df.isna().any(axis=1)]  # Verifica linhas com valores ausentes
print(f'Total de linhas com NaNs (nulos): {nans.shape[0]}\n')
nans

In [ ]:
# Remove os dados com descontinuidade de tempo
df = df['2017-03-29 12:00:00':]
df

O gráfico a seguir mostra o conteúdo mineral antes (isto é, na alimentação) e depois do processo de flotação (no concentrado). Como pode ser observado na figura, o propósito da flotação é aumentar a recuperação do mineral de ferro e reduzir a ganga (sílica).

Durante alguns períodos (por exemplo, de 13 de Maio a 13 de Junho), o conteúdo mineral na alimentação foi constante, mas o conteúdo resultante no concentrado variou. Isso sugere que a *%ferro* e *%sílica no concentrado* não são governados unicamente pelo conteúdo das matérias-primas, mas por outros parâmetros também (ou seja, variáveis de ambiente e processo).

In [ ]:
content = ['feed_iron', 'feed_silica', 'conc_iron', 'conc_silica']
palette = ['#FB6542', '#FFBB00', '#3F681C', '#375E97']

# Plota o conteúdo mineral antes e depois da flotação
plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(18,6))
for pct, color in zip(content, palette):
    ax.plot(df.index.values, pct, data=df, color=color)
ax.set_title('Conteúdo mineral na alimentação e no concentrado',
             loc='left', weight='bold', size=16)
ax.set_ylabel('% Mineral')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='center left')
plt.show()

A coluna `conc_iron` será removida do DataFrame porque nosso objetivo é analisar a previsão da *%sílica no concentrado* sem incluir a variável da *%ferro no concentrado* (já que as duas são extraídas juntas no fim do processo).

In [ ]:
cols = list(df)
cols.insert(0, cols.pop(         # Movendo a variável alvo `conc_silica` para a primeira posição
    cols.index('conc_silica')))  # Opcional, mas facilita a organização visual
df = df.loc[:, cols]
df.to_csv('./Flotation_Dataset_by_Hour.csv')  # Salva o dataset organizado

# Removemos a variável `conc_iron`
values = df.drop('conc_iron', axis=1).values

# Nota: A normalização (MinMaxScaler) foi movida para mais tarde no código, 
# APÓS separar os dados em treino e teste, para evitar Vazamento de Dados (Data Leakage).

O próximo estágio do pré-processamento envolve enquadrar (frame) o conjunto de dados como um problema de aprendizado supervisionado. Iremos prever a *%sílica no concentrado* na hora atual (*t*) dadas as outras características (ou seja, matérias-primas, ambiente, processo) dos passos de tempo anteriores (*t-n*). Transformaremos o conjunto de dados usando a função `series_to_supervised()` abaixo, que cria colunas defasadas (lags).

In [ ]:
# Converte a série em um aprendizado supervisionado (criando lags/defasagens de tempo)
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    """
    Transforma uma série temporal num conjunto de dados de aprendizado supervisionado.
    Argumentos:
        data: Sequência de observações como lista ou array NumPy.
        n_in: Número de observações passadas (lags) como entradas (X).
        n_out: Número de observações futuras como saída (y).
        dropnan: Se True, remove as linhas com valores NaN gerados pelas defasagens.
    Retorna:
        Pandas DataFrame com a série temporal formatada.
    """
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    
    for i in range(n_in, 0, -1):   # Sequência de entrada (t-n, ... t-1)
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
        
    for i in range(0, n_out):      # Sequência de previsão (t, t+1, ... t+n)
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
            
    agg = pd.concat(cols, axis=1)  # Concatena todas as colunas
    agg.columns = names
    if dropnan:                    # Remove linhas com valores nulos (resultantes do "shift")
        agg.dropna(inplace=True)
        
    # Remove as colunas no instante (t) que não queremos prever 
    # Queremos apenas prever var1(t) usando as variáveis em (t-1).
    drop_cols = ['var'+str(i)+'(t)' for i in range(2,23)]
    agg.drop(columns=drop_cols, axis=1, inplace=True)   
    return agg

In [ ]:
reframed = series_to_supervised(values, n_in=1, n_out=1)
reframed  # Exibe o dataset reenquadrado

As colunas `var1(t-1)` até `var22(t-1)` são nossos *features* (entradas). Os valores medidos das entradas são deslocados 1 hora para trás em relação à variável alvo `var1(t)` (*%sílica no concentrado* no momento *t*).

Os dados transformados serão divididos cronologicamente (sem embaralhar) em três conjuntos: treino (60%), validação (20%) e teste (20%). Usamos a divisão cronológica pois em séries temporais, a ordem dos dados é essencial e modelos aprendem melhor a prever o "futuro" apenas com informações do "passado".

In [ ]:
n_features = 22             # Número de variáveis independentes (entradas)
n_hours = 1                 # Número de horas de defasagem (lag)
n_obs = n_hours * n_features

# Define o tamanho de linhas para cada divisão (Treino/Validação/Teste)
n_train = int(np.round(len(reframed)*.60))
n_valid = int(np.round(len(reframed)*.20))
n_test = int(np.round(len(reframed)*.20))

# Separação sequencial dos dados
dataset_values = reframed.values
train = dataset_values[:n_train, :]
valid = dataset_values[n_train:(n_train+n_valid), :]
test = dataset_values[(n_train+n_valid):, :]

# Separa as variáveis independentes (X) da variável alvo (y)
train_X, train_y = train[:, :n_obs], train[:, -1]
valid_X, valid_y = valid[:, :n_obs], valid[:, -1]
test_X, test_y = test[:, :n_obs], test[:, -1]

# Normalização usando MinMaxScaler APÓS o split para EVITAR vazamento de dados
# O scaler vai "aprender" (fit) apenas os limites numéricos do conjunto de treino.
scaler_X = MinMaxScaler(feature_range=(0, 1))
train_X = scaler_X.fit_transform(train_X)
valid_X = scaler_X.transform(valid_X)
test_X = scaler_X.transform(test_X)

# Vamos normalizar a variável alvo Y separadamente para depois podermos inverter sua escala facilmente
scaler_y = MinMaxScaler(feature_range=(0, 1))
train_y = scaler_y.fit_transform(train_y.reshape(-1, 1)).flatten()
valid_y = scaler_y.transform(valid_y.reshape(-1, 1)).flatten()
test_y = scaler_y.transform(test_y.reshape(-1, 1)).flatten()

# Redimensiona os dados de entrada (X) para serem tensores 3D [amostras, timesteps, variáveis] para o modelo LSTM
train_X = train_X.reshape((train_X.shape[0], n_hours, n_features))
valid_X = valid_X.reshape((valid_X.shape[0], n_hours, n_features))
test_X = test_X.reshape((test_X.shape[0], n_hours, n_features))

print(  # Exibe o shape final de cada conjunto
    f'Conjunto de Treino    : {train_X.shape}, {train_y.shape}',
    f'\nConjunto de Validação : {valid_X.shape}, {valid_y.shape}',
    f'\nConjunto de Teste     : {test_X.shape}, {test_y.shape}')

# Construção e Treinamento do Modelo

Prever ganga em concentrados de flotação é um problema de séries temporais. Uma variação das redes neurais recorrentes (RNN), chamada memória de longo e curto prazo (*Long Short-Term Memory* - LSTM), é uma arquitetura de *Deep Learning* excelente para esse problema.

> *LSTMs levam vantagem sobre as redes neurais tradicionais (feed-forward) e RNNs comuns por sua capacidade de lembrar seletivamente padrões e dependências temporais longas.*

A construção e o treinamento do modelo são feitos usando o framework [PyTorch](https://pytorch.org/). O modelo é formado por duas camadas LSTM, com 16 neurônios/células de memória, seguidas por uma camada totalmente conectada (Linear) para a saída contínua de regressão. Durante o treino, usamos lotes de 16 amostras (batch) ao longo de 100 *epochs* (passagens completas pelos dados de treino).

In [ ]:
# Converte os arrays do NumPy para tensores do PyTorch
train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
train_y_tensor = torch.tensor(train_y, dtype=torch.float32).view(-1, 1)
valid_X_tensor = torch.tensor(valid_X, dtype=torch.float32)
valid_y_tensor = torch.tensor(valid_y, dtype=torch.float32).view(-1, 1)
test_X_tensor = torch.tensor(test_X, dtype=torch.float32)

# Cria os DataLoaders para gerenciar os batches
train_dataset = TensorDataset(train_X_tensor, train_y_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)

# Define a classe do modelo LSTM em PyTorch
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # Inicializa o hidden state e o cell state com zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # Passa pela LSTM
        out, _ = self.lstm(x, (h0, c0))
        
        # Pega a saída do último time step
        out = self.fc(out[:, -1, :])
        return out

input_size = train_X.shape[2] # n_features
hidden_size = 16
num_layers = 2
output_size = 1

model = LSTMModel(input_size, hidden_size, num_layers, output_size)
print(model)

# Define a função de perda (L1Loss equivale ao MAE) e o otimizador ADAM
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 100
train_loss = []
valid_loss = []
best_valid_loss = float('inf')

for epoch in range(epochs):
    model.train()
    batch_losses = []
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
        
    # Calcula a perda média do treino para esta epoch
    avg_train_loss = np.mean(batch_losses)
    train_loss.append(avg_train_loss)
    
    # Avaliação no conjunto de validação
    model.eval()
    with torch.no_grad():
        valid_pred = model(valid_X_tensor)
        v_loss = criterion(valid_pred, valid_y_tensor)
        valid_loss.append(v_loss.item())
        
        # Salva o melhor modelo simulando o ModelCheckpoint
        if v_loss.item() < best_valid_loss:
            best_valid_loss = v_loss.item()
            torch.save(model.state_dict(), './LSTM_Flotation_Gangue.pth')
            
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - loss: {avg_train_loss:.4f} - val_loss: {v_loss.item():.4f}")

In [ ]:
# Extrai as perdas (losses) do histórico de treinamento
train_loss = history.history['loss']
valid_loss = history.history['val_loss']

# Plota a curva de aprendizado
plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(9,6))
ax.plot(train_loss, color=palette[0], label='Treinamento (Training)')
ax.plot(valid_loss, color=palette[2], label='Validação (Validation)')
ax.set_title('Curvas de Aprendizado (Learning Curves)', loc='left', weight='bold', size=16)
ax.set_xlabel('Epoch (Passagem)')
ax.set_ylabel('Loss / Perda (Erro Absoluto Médio)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='center right')
plt.show()

O gráfico acima indica que o modelo encontrou um bom ajuste, já que as linhas de perda (*loss*) do treinamento e validação caem e estabilizam juntas, com um "gap" muito pequeno entre as duas, demonstrando que o modelo não está sofrendo um forte sobredimensionamento (*overfitting*).

In [ ]:
print(f'Melhor loss no Treinamento = {min(train_loss):.4f}',
      f'na epoch {train_loss.index(min(train_loss))}',
      f'\nMelhor loss na Validação = {min(valid_loss):.4f}',
      f'na epoch {valid_loss.index(min(valid_loss))}')

# Realizando as Previsões com o modelo LSTM

A previsão agora é realizada usando o modelo treinado nos **dados de teste** (que permaneceram intocados até esse momento). Tanto as previsões do modelo quanto os valores reais são invertidos de volta às suas escalas originais (removendo a normalização `MinMaxScaler`) antes de calcular as métricas finais de erro. As duas principais métricas de erro que usamos são: Erro Absoluto Médio (MAE) e a Raiz do Erro Quadrático Médio (RMSE).

In [ ]:
# Carrega os pesos do melhor modelo salvo
model.load_state_dict(torch.load('./LSTM_Flotation_Gangue.pth'))
model.eval()

# Realiza as predições usando os dados de teste intocados
with torch.no_grad():
    yhat_tensor = model(test_X_tensor)
    yhat = yhat_tensor.numpy()

# Inverte a escala para as predições (traz os valores normalizados de volta para os limites reais)
inv_yhat = scaler_y.inverse_transform(yhat).flatten()

# Inverte a escala para os valores reais de teste
inv_y = scaler_y.inverse_transform(test_y.reshape(-1, 1)).flatten()

# Calcula as métricas de erro
mae = mean_absolute_error(inv_y, inv_yhat)
rmse = np.sqrt(mean_squared_error(inv_y, inv_yhat))
print(32*'-'+'\nAVALIAÇÃO DA PREVISÃO'+'\n'+32*'-',
      f'\nErro Absoluto Médio (MAE)     : {mae:.4f}',
      f'\nRaiz do Erro Quadrático Médio : {rmse:.4f}')

In [ ]:
# Define os dias no eixo-X
test_date = df.index[-test_y.shape[0]:]

# Calcula o intervalo de confiança de 95% (Valor-Z = 1.96)
ci = 1.96 * np.std(inv_y) / np.mean(inv_y)

# Plota os valores reais e as predições do modelo
plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(18,6))
ax.plot(test_date, inv_y, color=palette[1], label='Valor Real (Actual)')
ax.plot(test_date, inv_yhat, color=palette[2], label='Previsão do Modelo (Forecast)')
ax.fill_between(test_date, (inv_y-ci), (inv_y+ci), color=palette[3],
                alpha=.1, label='Intervalo de Confiança (95%)')
ax.set_title('% Sílica no Concentrado: Valores Reais vs Previsão do LSTM',
             loc='left', weight='bold', size=16)
ax.set_ylabel('Sílica no Concentrado (%)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend()
plt.show()

A comparação entre as previsões e os valores reais da *%sílica no concentrado* é mostrada acima para o período de testes (Agosto até Setembro de 2017). As previsões seguem muito de perto o padrão geral das oscilações de valores reais e são bem comportadas de acordo com o intervalo de confiança.

#### Comparação de Previsão com Regressor Random Forest
Um regressor de Florestas Aleatórias (Random Forest), em comparação com o LSTM, muitas vezes tem dificuldades em explorar dependências temporais longas com atrasos simples, podendo gerar erros ligeiramente maiores na extrapolação.

In [ ]:
# Reenquadra (reframe) os dados adicionando uma defasagem de tempo de 1 hora
rf_values = df.drop('conc_iron', axis=1).values
rf_reframed = series_to_supervised(rf_values, n_in=1, n_out=1)

# Extrai os atributos previsores (X) e o objetivo (y)
rf_X = rf_reframed.values[:, :-1]
rf_y = rf_reframed.values[:, -1]

# Divide os dados de forma sequencial (não embaralhada) em proporção de 80% treino, 20% teste
rf_train_X, rf_test_X, rf_train_y, rf_test_y = train_test_split(
    rf_X, rf_y, test_size=.20, random_state=0, shuffle=False)

# Normaliza as colunas de entrada (evitando Data Leakage porque dividimos primeiro)
rf_scaler = MinMaxScaler(feature_range=(0,1))
rf_train_X = rf_scaler.fit_transform(rf_train_X)
rf_test_X = rf_scaler.transform(rf_test_X)

# Instancia o regressor Random Forest
forest = RandomForestRegressor(random_state=0)

# Realiza o treinamento
forest.fit(rf_train_X, rf_train_y)

# Realiza previsões no conjunto de teste
rf_yhat = forest.predict(rf_test_X)

# Calcula os erros
rf_mae = mean_absolute_error(rf_test_y, rf_yhat)
rf_rmse = np.sqrt(mean_squared_error(rf_test_y, rf_yhat))
print(32*'-'+'\nAVALIAÇÃO DA PREVISÃO'+'\n'+32*'-',
      f'\nErro Absoluto Médio (MAE)     : {rf_mae:.4f}',
      f'\nRaiz do Erro Quadrático Médio : {rf_rmse:.4f}')

In [ ]:
# Plota os valores reais comparados ao Random Forest
plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(18,6))
ax.plot(test_date, rf_test_y, color=palette[1], label='Valor Real (Actual)')
ax.plot(test_date, rf_yhat, color=palette[2], label='Previsão Random Forest (Forecast)')
ax.fill_between(test_date, (rf_test_y-ci), (rf_test_y+ci), color=palette[3],
                alpha=.1, label='Intervalo de Confiança (95%)')
ax.set_title('% Sílica no Concentrado: Valores Reais vs Previsão por Random Forest',
             loc='left', weight='bold', size=16)
ax.set_ylabel('Sílica no Concentrado (%)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend()
plt.show()

# Conclusão
Uma abordagem de *Deep Learning* utilizando Redes Neurais Recorrentes LSTM foi implementada com sucesso para prever as concentrações de ganga (sílica) em um processo de flotação mineral. Excluindo a *%ferro no concentrado* dos atributos do modelo, a *%sílica no concentrado* pôde ser prevista uma hora à frente com o erro inferior a 1% (com base em RMSE e MAE). 

Sendo considerado que na mineração real um desvio MAE e RMSE em torno de 1±0.2 é muito satisfatório, essas previsões apresentam uma ferramenta muito promissora. Elas viabilizam que engenheiros de processo façam avaliações preventivas e tomem medidas ativas (ex.: controle de ar nas células ou ajuste da dosagem de amina) antes da análise real de laboratório estar concluída, evitando penalidades comerciais no concentrado.

Finalmente, embora esse notebook atinja seus propósitos, existem opções para exploração futura:
- Previsões com um *lag time* menor. Por exemplo, criando defasagens de tempo na ordem de 30 minutos em vez de 1 hora.
- Análise de Importância de Atributos (Feature Importance) para interpretar corretamente quais variáveis influenciam com mais peso e força na variância de sílica final, permitindo ajustes focados na fábrica.